# Gemma 3 270M — NER Pipeline

**Flow:** Login → Data → Tokenizer → Model → Dry run → Mini train → Predict

> Run cells top-to-bottom. GPU recommended but CPU works for dry run.

## 1. HuggingFace Login
Gemma is a gated model — login once and the token is cached for all future runs.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Imports & Config

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))   # make src/ importable

import torch
from transformers import AutoTokenizer
from config import Config
from src.model import GemmaBackboneNER
from src.data import read_conll, make_loader

cfg    = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"torch      : {torch.__version__}")
print(f"device     : {device}")
print(f"model_id   : {cfg.model.gemma_model_id}")
print(f"max_seq_len: {cfg.model.max_seq_len}")

## 3. Data Exploration

In [ ]:
train_samples = read_conll("data/train.txt")
val_samples   = read_conll("data/valid.txt")
test_samples  = read_conll("data/test.txt")

print(f"Train : {len(train_samples):,} sentences")
print(f"Val   : {len(val_samples):,} sentences")
print(f"Test  : {len(test_samples):,} sentences")

# Show a sample
s = train_samples[0]
print(f"\nSample 0 — {len(s.words)} words")
print(f"{'Word':<20} {'Identity':<12} {'Location':<12} {'Temporal':<12} {'Domain'}")
print("-" * 70)
for w, i, l, t, d in zip(s.words, s.identity, s.location, s.temporal, s.domain):
    print(f"{w:<20} {i:<12} {l:<12} {t:<12} {d}")

## 4. Tokenizer & DataLoader

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model.gemma_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size : {tokenizer.vocab_size:,}")
print(f"Pad token  : '{tokenizer.pad_token}'")

# Build loaders (use small batch for dry run)
train_loader = make_loader(train_samples, tokenizer, cfg.model.max_seq_len, batch_size=4, shuffle=True)
val_loader   = make_loader(val_samples,   tokenizer, cfg.model.max_seq_len, batch_size=4, shuffle=False)

batch = next(iter(train_loader))
print(f"\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k:<22} {tuple(v.shape)}")

## 5. Model — Instantiation & Parameter Count

In [ ]:
model = GemmaBackboneNER(cfg.model).to(device)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Total params     : {total:>12,}")
print(f"Frozen  (Gemma)  : {frozen:>12,}")
print(f"Trainable        : {trainable:>12,}  ← BiLSTM + 4 CRF heads")
print(f"Train %          : {100 * trainable / total:.2f}%")
print(f"Device           : {device}")

## 6. Dry Run — Forward + Backward Pass

In [ ]:
batch_dev = {k: v.to(device) for k, v in batch.items()}

# Forward
loss, h = model(**batch_dev)
print(f"Loss  : {loss.item():.4f}")
print(f"h     : {tuple(h.shape)}  (batch, seq_len, bilstm_out=512)")

# Backward
loss.backward()

# Gradient check
gemma_grads  = sum(1 for p in model.gemma.parameters()  if p.grad is not None)
bilstm_grads = sum(1 for p in model.bilstm.parameters() if p.grad is not None)
print(f"\nGradient check:")
print(f"  Gemma  params with grad : {gemma_grads}  (should be 0 — frozen)")
print(f"  BiLSTM params with grad : {bilstm_grads}  (should be > 0)")
print("\n✓ Dry run complete" if gemma_grads == 0 and bilstm_grads > 0 else "\n✗ Something is wrong")

## 7. Mini Training Loop (20 steps)
Sanity check that loss decreases before committing to full training.

In [ ]:
from torch.optim import AdamW

# Fresh model + optimizer for mini loop
model.zero_grad()
optimizer = AdamW(model.trainable_params(), lr=cfg.train.learning_rate, weight_decay=cfg.train.weight_decay)

model.train()
losses = []

for step, batch in enumerate(train_loader):
    if step >= 20:
        break
    batch = {k: v.to(device) for k, v in batch.items()}
    loss, _ = model(**batch)
    if loss is None:
        continue
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.trainable_params(), cfg.train.grad_clip)
    optimizer.step()
    losses.append(loss.item())
    print(f"  step {step+1:02d}  loss={loss.item():.4f}")

print(f"\nFirst loss : {losses[0]:.4f}")
print(f"Last  loss : {losses[-1]:.4f}")
print(f"Trend      : {'↓ decreasing ✓' if losses[-1] < losses[0] else '↑ not yet decreasing (normal for 20 steps)'}")

## 8. Sample Prediction
Run inference on a few example sentences to see entity spans.

In [ ]:
from src.span_merger import decode_batch

model.eval()

test_sentences = [
    "Alice Johnson signed a contract with Anthropic in San Francisco on Monday.",
    "The GDPR regulation was discussed at the World Economic Forum in Davos.",
    "Bob Martinez will attend the Summer Olympics in Paris next July.",
]

for text in test_sentences:
    words = text.split()
    enc   = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=cfg.model.max_seq_len,
        return_tensors="pt",
    )
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.inference_mode():
        ident, loc, temp, dom, h = model.predict(input_ids, attention_mask)
        heads    = [model.identity_head, model.location_head, model.temporal_head, model.domain_head]
        entities = decode_batch((ident, loc, temp, dom), h, heads, attention_mask)

    print(f"\n>>> {text}")
    if not entities[0]:
        print("  (no entities detected)")
    for e in entities[0]:
        span_words = " ".join(words[e.start:e.end])
        print(f"  [{e.start}:{e.end}] '{span_words}'  →  {e.label:<28}  conf={e.confidence:.3f}")